In [1]:
import pandas as pd

path = "../data/raw/"
orders = pd.read_csv(path + "olist_orders_dataset.csv")

In [2]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for c in date_cols:
    orders[c] = pd.to_datetime(orders[c])

orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [3]:
orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [4]:
print("Fully duplicated rows:", orders.duplicated().sum())
print("Duplicated order_id:", orders["order_id"].duplicated().sum())

Fully duplicated rows: 0
Duplicated order_id: 0


In [5]:
orders[orders["order_delivered_customer_date"].isnull()]["order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [6]:
delivered = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].notnull())
].copy()

print("Delivered orders with a delivery date:", len(delivered))

Delivered orders with a delivery date: 96470


In [7]:
delivered["delivery_days"] = (
    delivered["order_delivered_customer_date"] - delivered["order_purchase_timestamp"]
).dt.days

delivered["delay_days"] = (
    delivered["order_delivered_customer_date"] - delivered["order_estimated_delivery_date"]
).dt.days

delivered["is_late"] = delivered["order_delivered_customer_date"] > delivered["order_estimated_delivery_date"]

delivered["order_month"] = delivered["order_purchase_timestamp"].dt.strftime("%Y-%m")

delivered[["delivery_days", "delay_days", "is_late", "order_month"]].head()

,delivery_days,delay_days,is_late,order_month
0,8,-8,False,2017-10
1,13,-6,False,2018-07
2,9,-18,False,2018-08
3,13,-13,False,2017-11
4,2,-10,False,2018-02


In [8]:
late_pct = delivered["is_late"].mean() * 100
print(f"Late deliveries: {late_pct:.1f}%")
print("Average delivery time (days):", round(delivered["delivery_days"].mean(), 1))

Late deliveries: 8.1%
Average delivery time (days): 12.1


In [9]:
import os
os.makedirs("../data/processed", exist_ok=True)
delivered.to_csv("../data/processed/orders_clean.csv", index=False)

In [10]:
items = pd.read_csv(path + "olist_order_items_dataset.csv")
print("Total price:", round(items["price"].sum(), 2))

Total price: 13591643.7


In [11]:
reviews = pd.read_csv(path + "olist_order_reviews_dataset.csv")
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])

print("Rows:", len(reviews))
print("Orders with more than one review:", reviews["order_id"].duplicated().sum())

reviews = (reviews.sort_values("review_answer_timestamp")
                  .drop_duplicates(subset="order_id", keep="last"))
print("Rows after keeping the latest review per order:", len(reviews))

Rows: 99224
Orders with more than one review: 551
Rows after keeping the latest review per order: 98673


In [12]:
merged = delivered.merge(reviews[["order_id", "review_score"]], on="order_id", how="left")
print(merged.shape)
merged["review_score"].isnull().sum()

(96470, 13)


np.int64(646)

In [13]:
merged.groupby("is_late")["review_score"].agg(["mean", "count"]).round(2)

,mean,count
is_late,,
False,4.29,88163
True,2.57,7661


In [14]:
merged.to_csv("../data/processed/orders_reviews_clean.csv", index=False)